In [2]:
from dotenv import load_dotenv
import os

load_dotenv() 

True

In [73]:
!pip install wikipedia chromadb faiss-cpu langchain-classic langchain-community langchain-huggingface huggingface_hub 

  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
Using cached accelerate-1.13.0-py3-none-any.whl (383 kB)


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from langchain_community.retrievers import WikipediaRetriever

In [6]:
# Initialize the retriever (optional: set language and top_k)
retriever = WikipediaRetriever(language="en", top_k_results=2)

In [7]:
# Define your query
query = "India Pakistan relations"

# Get relevant documents from Wikipedia
doc = retriever.invoke(query)

In [8]:
# Print retieved content
for i, doc in enumerate(doc):
    print(f"\n-- Result {i+1} --")
    print(f"Content:\n{doc.page_content}...") # truncate for display


-- Result 1 --
Content:
India and Pakistan have a complex and largely hostile relationship that is rooted in a multitude of historical and political events, most notably the partition of British India in August 1947.
Two years after World War II, the United Kingdom formally dissolved British India, dividing it into two new sovereign nations: the Union of India and Pakistan. The partitioning of the former British colony resulted in the displacement of up to 15 million people, with the death toll estimated to have reached between several hundred thousand and one million people as Hindus and Muslims migrated in opposite directions across the Radcliffe Line to reach India and Pakistan, respectively. In 1950, India emerged as a secular republic with a Hindu-majority population. Shortly afterwards, in 1956, Pakistan emerged as an Islamic republic with a Muslim-majority population.
While the two South Asian countries established full diplomatic ties shortly after their formal independence, t

Vector Store Retriever

In [9]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

c:\Users\yckar\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
# Step 1: Your source documents
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [11]:
# Step 2: Initialize an embedding model
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Step 3: Create a Chroma vector store in memory
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name="my_collection",
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15138.71it/s]


In [12]:
# Step 4: Convert vectorstore into a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [13]:
query = "What is chroma use for?"
result = retriever.invoke(query)

In [14]:
for i, doc in enumerate(result):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
LangChain helps developers build LLM applications easily.


In [15]:
results =vectorstore.similarity_search(query, k=2)

In [16]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
LangChain helps developers build LLM applications easily.


MMR

In [17]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [18]:
from langchain_community.vectorstores import FAISS

 # Step 1:Initialize HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Step 2: Create a FAISS vector store from documents
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embedding_model,
)



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15234.80it/s]


In [25]:
retriever = vectorstore.as_retriever(
    search_type="mmr",  # < -- This enables MMR search
    search_kwargs={"k": 3, "lambda_mult": 1} # k = top results, lambda_mult = relevance-diversity balance
)

In [26]:
query = "What is langchain used for?"
results  = retriever.invoke(query)

In [27]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
LangChain is used to build LLM based applications.

--- Result 2 ---
LangChain supports Chroma, FAISS, Pinecone, and more.

--- Result 3 ---
LangChain makes it easy to work with LLMs.


Multi-Query Retriever

In [78]:
!pip install transformers accelerate


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from transformers import pipeline
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings,HuggingFaceEndpoint,HuggingFacePipeline
from langchain_core.documents import Document
from langchain_classic.retrievers import MultiQueryRetriever

c:\Users\yckar\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [4]:
# Initialize embedding model
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Create a FAISS vector store from documents
vectorstore = FAISS.from_documents(all_docs, embedding=embedding_model)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13614.44it/s]


In [15]:
# model that can generate multiple queries for a given input question
from langchain_huggingface import HuggingFaceEndpoint,ChatHuggingFace
model = ChatHuggingFace(llm=HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-V4-Pro", 
    task="text-generation",
         
))


In [17]:
# Create retrievers
similarity_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [18]:
multiquery_retriever = MultiQueryRetriever.from_llm(retriever=vectorstore.as_retriever(search_kwargs={"k": 5}),llm=model)

In [25]:
# Query
query = "How to improve energy levels and maintain balance? "

In [26]:
similarity_results = similarity_retriever.invoke(query)
multiquery_results = multiquery_retriever.invoke(query)

In [27]:
for i, doc in enumerate(similarity_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

print("*"*150)

for i, doc in enumerate(multiquery_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
The solar energy system in modern homes helps balance electricity demand.

--- Result 3 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

--- Result 4 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 5 ---
Photosynthesis enables plants to produce energy by converting sunlight.
******************************************************************************************************************************************************

--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

--- Result 3 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 4 ---
The solar energy system in modern homes helps balance elect

Contextual Compression Retiever

In [28]:

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings,HuggingFaceEndpoint,HuggingFacePipeline
from langchain_core.documents import Document
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_classic.retrievers import ContextualCompressionRetriever

In [29]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [30]:
# Create a FAISS vector store from documents
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embedding_model)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12990.15it/s]


In [31]:
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [32]:
# Set up the LLM for extraction
# model that can generate multiple queries for a given input question
from langchain_huggingface import HuggingFaceEndpoint,ChatHuggingFace
model = ChatHuggingFace(llm=HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-V4-Pro", 
    task="text-generation",
         
))
compressor = LLMChainExtractor.from_llm(model)


In [33]:
# Create the contextual compression retriever
compression_retriever = ContextualCompressionRetriever(base_compressor=compressor, base_retriever=base_retriever)

In [34]:
query = "What is photosynthesis?"
compression_results = compression_retriever.invoke(query)

In [35]:
for i, doc in enumerate(compression_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Photosynthesis is the process by which green plants convert sunlight into energy.

--- Result 2 ---
The chlorophyll in plant cells captures sunlight during photosynthesis.
